<a href="https://colab.research.google.com/github/BostonM-Curtin/ISYS2001-BostonM/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Lab Ticket: Budget Decisions

This week your job is not to hand-code a program. It is to write **one complete, detailed prompt** that an AI can turn into a working budget tool in a single go.

In the worksheet you practised giving AI clear intent. Here that is the whole task. The skill being tested is whether you can describe exactly what you want, in enough detail, that one prompt produces the finished tool with no back-and-forth.

## What to submit

1. **Your final prompt**: one complete, self-contained prompt (paste it into the prompt cell below).
2. **The code it produced**: paste the AI-generated program into the code cell and run it, so you can show it works.

Your prompt can be the result of several rounds of refining. What you hand in is the single finished version: the one that would generate the whole tool if someone ran it cold.

## The brief

Design a tool that helps someone make a smart budget decision. What that means, and who it is for, is up to you. Some directions, though you are not limited to these:

- An expense classifier that sorts spending into categories and reacts to each.
- A budget checker that warns when an expense is too large a share of someone's budget.
- A savings goal tracker that gives different feedback depending on progress.
- A purchase advisor that weighs a price against how much money and income the person has.

Pick one or invent your own. The only hard requirement is that the finished program makes genuinely different decisions depending on the numbers it is given.

## How this works

You are going to write a prompt, hand it to an AI, and see what it builds. Your first attempt will not be right. Read what comes back and look for the gaps:

- Did it invent a rule you did not ask for? Your prompt was not precise enough.
- Did it miss a case? Say so explicitly next time.
- Did it ask you follow-up questions? A complete prompt should not need any.

Fold each fix back into the prompt and run it again. Keep going until one single prompt produces the whole working tool in one go. That final version is what you submit.

How detailed does it need to be? Detailed enough that a classmate could paste it cold and get the same working tool. Work out for yourself what that takes.

## Your final prompt

> Paste your complete, finished prompt below (double-click to edit).

```
Write a complete, working Python program: a Smart Purchase Advisor.

CONTEXT
The tool helps a student decide whether a planned purchase is a smart financial move, based on their income, existing expenses, and savings goal.

INPUTS (collect via input(), in this order)
1. Monthly income (float)
2. Monthly fixed expenses — rent, subscriptions, etc. (float)
3. Current savings balance (float)
4. Savings goal amount (float)
5. Price of the item they want to buy (float)

Validate every numeric input: reject negative numbers and non-numeric text, re-prompting until valid input is given.

LOGIC
1. Calculate disposable income = monthly income − fixed expenses.
2. Calculate the purchase's percentage of disposable income = (price / disposable income) × 100. If disposable income is 0 or negative, skip this calculation and treat the purchase as automatically high-risk.
3. Calculate progress toward the savings goal = (current savings / savings goal) × 100, capped for display at 100%.
4. Classify the purchase into exactly one of four decisions, checked in this order:
   - "NOT RECOMMENDED": disposable income ≤ 0, OR price > disposable income.
   - "RISKY": price is more than 50% of disposable income.
   - "THINK TWICE": price is between 25% and 50% of disposable income, OR savings goal progress is below 50%.
   - "GO FOR IT": price is 25% or less of disposable income AND savings goal progress is 50% or higher.
5. Each decision must produce a DIFFERENT, tailored multi-line message: explain the reasoning using the actual numbers calculated (not generic text), and give one concrete piece of advice specific to that decision level.

OUTPUT
Print a formatted summary showing: disposable income, purchase as % of disposable income, savings goal progress %, the decision, and the tailored message. Use clear section headers and aligned currency formatting (2 decimal places, $ sign).

STRUCTURE
- Use functions: one to get and validate a single numeric input (reusable, takes a prompt and a label for error messages), one to run the calculations and return the results, one to determine the decision, one to build the tailored message, and one main() that ties it together.
- Add a docstring at the top of the file explaining what the program does.
- Add inline comments explaining each major step.

TESTING
After the function definitions, include a commented-out block of at least 3 sample test cases (as direct function calls, not input()) showing different inputs that produce different decisions, so the branching logic can be verified without manual input.

Do not include any explanation before or after the code — output only the complete Python program```

In [1]:
"""
Smart Purchase Advisor
-----------------------
This program helps a student decide whether a planned purchase is a smart
financial move. It takes the user's monthly income, fixed expenses, current
savings, savings goal, and the price of an item they want to buy. It then
calculates disposable income, the purchase's share of that disposable
income, and progress toward the savings goal. Based on these numbers, it
classifies the purchase into one of four decision levels (NOT RECOMMENDED,
RISKY, THINK TWICE, GO FOR IT) and prints a tailored explanation with
concrete advice.
"""


def get_valid_number(prompt, label):
    """
    Repeatedly prompt the user until they enter a valid, non-negative number.
    'prompt' is the text shown to the user.
    'label' is used in error messages to identify which value is being asked for.
    Returns the value as a float.
    """
    while True:
        raw_value = input(prompt)
        try:
            value = float(raw_value)
        except ValueError:
            print(f"  -> Invalid input for {label}. Please enter a number (e.g. 250 or 250.50).")
            continue

        if value < 0:
            print(f"  -> {label} cannot be negative. Please try again.")
            continue

        return value


def calculate_results(income, fixed_expenses, savings, savings_goal, price):
    """
    Perform all calculations needed for the decision:
    - disposable income
    - purchase price as a percentage of disposable income
    - savings goal progress percentage (capped at 100 for display)
    Returns a dictionary of these results.
    """
    # Disposable income is what's left after fixed expenses are paid
    disposable_income = income - fixed_expenses

    # If there's no disposable income (zero or negative), the purchase
    # percentage calculation is meaningless/undefined, so we skip it and
    # flag it directly as automatically high-risk with a sentinel value.
    if disposable_income <= 0:
        purchase_percentage = None  # signals "cannot calculate / high risk"
    else:
        purchase_percentage = (price / disposable_income) * 100

    # Savings goal progress; guard against divide-by-zero if goal is 0
    if savings_goal > 0:
        goal_progress = (savings / savings_goal) * 100
    else:
        goal_progress = 100.0  # no goal set means "goal" is trivially met

    # Cap the displayed progress at 100%, but keep the raw value for logic
    goal_progress_display = min(goal_progress, 100.0)

    return {
        "disposable_income": disposable_income,
        "purchase_percentage": purchase_percentage,
        "goal_progress": goal_progress,
        "goal_progress_display": goal_progress_display,
        "price": price,
    }


def determine_decision(results):
    """
    Classify the purchase into exactly one of four decision levels,
    checked in strict order: NOT RECOMMENDED -> RISKY -> THINK TWICE -> GO FOR IT.
    """
    disposable_income = results["disposable_income"]
    purchase_percentage = results["purchase_percentage"]
    goal_progress = results["goal_progress"]
    price = results["price"]

    # NOT RECOMMENDED: no disposable income at all, or price exceeds it outright
    if disposable_income <= 0 or price > disposable_income:
        return "NOT RECOMMENDED"

    # From here on, disposable_income > 0 and purchase_percentage is a real number
    if purchase_percentage > 50:
        return "RISKY"

    if 25 <= purchase_percentage <= 50 or goal_progress < 50:
        return "THINK TWICE"

    # purchase_percentage <= 25 AND goal_progress >= 50
    return "GO FOR IT"


def build_message(decision, results):
    """
    Build a tailored, multi-line message for the given decision, using the
    actual calculated numbers so the explanation is specific to this case.
    """
    disposable_income = results["disposable_income"]
    purchase_percentage = results["purchase_percentage"]
    goal_progress_display = results["goal_progress_display"]
    price = results["price"]

    if decision == "NOT RECOMMENDED":
        if disposable_income <= 0:
            reason = (
                f"Your disposable income is ${disposable_income:.2f}, meaning your fixed\n"
                f"  expenses already use up all (or more than) your income."
            )
        else:
            reason = (
                f"This item costs ${price:.2f}, which is more than your entire\n"
                f"  disposable income of ${disposable_income:.2f}."
            )
        message = (
            f"{reason}\n"
            f"  Advice: Hold off on this purchase entirely. Focus on increasing income\n"
            f"  or reducing fixed expenses before considering non-essential spending."
        )

    elif decision == "RISKY":
        message = (
            f"This purchase would use {purchase_percentage:.1f}% of your ${disposable_income:.2f}\n"
            f"  disposable income - more than half of what you have left each month.\n"
            f"  Advice: Consider waiting until you've saved specifically for this item,\n"
            f"  rather than covering it from a single month's disposable income."
        )

    elif decision == "THINK TWICE":
        message = (
            f"This purchase would use {purchase_percentage:.1f}% of your ${disposable_income:.2f}\n"
            f"  disposable income, and your savings goal progress is currently\n"
            f"  {goal_progress_display:.1f}%.\n"
            f"  Advice: It's affordable, but weigh it against your savings goal first -\n"
            f"  even a short delay could get your progress above 50% before you buy."
        )

    else:  # GO FOR IT
        message = (
            f"This purchase only uses {purchase_percentage:.1f}% of your ${disposable_income:.2f}\n"
            f"  disposable income, and you're already {goal_progress_display:.1f}% of the way\n"
            f"  to your savings goal.\n"
            f"  Advice: You're in a solid position - this purchase fits comfortably\n"
            f"  within your budget without derailing your savings progress."
        )

    return message


def print_summary(results, decision, message):
    """
    Print a clearly formatted summary of the calculations, decision, and
    tailored message, with aligned currency formatting.
    """
    disposable_income = results["disposable_income"]
    purchase_percentage = results["purchase_percentage"]
    goal_progress_display = results["goal_progress_display"]
    price = results["price"]

    print("\n" + "=" * 50)
    print("SMART PURCHASE ADVISOR - SUMMARY")
    print("=" * 50)
    print(f"{'Item Price:':<30}${price:>10.2f}")
    print(f"{'Disposable Income:':<30}${disposable_income:>10.2f}")

    if purchase_percentage is None:
        print(f"{'Purchase % of Disposable Income:':<30}{'N/A (no disposable income)':>15}")
    else:
        print(f"{'Purchase % of Disposable Income:':<30}{purchase_percentage:>10.1f}%")

    print(f"{'Savings Goal Progress:':<30}{goal_progress_display:>10.1f}%")
    print("-" * 50)
    print(f"DECISION: {decision}")
    print("-" * 50)
    print(message)
    print("=" * 50 + "\n")


def main():
    """
    Main program flow: collect inputs, run calculations, determine the
    decision, build the tailored message, and print the summary.
    """
    print("Welcome to the Smart Purchase Advisor!")
    print("Answer the following questions to see if your purchase is a smart move.\n")

    # Step 1: Collect and validate all numeric inputs
    income = get_valid_number("Enter your monthly income: $", "Monthly income")
    fixed_expenses = get_valid_number("Enter your monthly fixed expenses: $", "Fixed expenses")
    savings = get_valid_number("Enter your current savings balance: $", "Current savings")
    savings_goal = get_valid_number("Enter your savings goal amount: $", "Savings goal")
    price = get_valid_number("Enter the price of the item you want to buy: $", "Item price")

    # Step 2: Run calculations
    results = calculate_results(income, fixed_expenses, savings, savings_goal, price)

    # Step 3: Determine the decision level
    decision = determine_decision(results)

    # Step 4: Build the tailored message for that decision
    message = build_message(decision, results)

    # Step 5: Print the formatted summary
    print_summary(results, decision, message)


# ---------------------------------------------------------------------------
# SAMPLE TEST CASES (uncomment to run without manual input)
# These call the functions directly with fixed values to verify that
# different inputs produce different decisions.
# ---------------------------------------------------------------------------

# def run_test(label, income, fixed_expenses, savings, savings_goal, price):
#     print(f"\n--- TEST CASE: {label} ---")
#     results = calculate_results(income, fixed_expenses, savings, savings_goal, price)
#     decision = determine_decision(results)
#     message = build_message(decision, results)
#     print_summary(results, decision, message)
#
# # Test 1: No disposable income at all -> NOT RECOMMENDED
# run_test("No disposable income", income=2000, fixed_expenses=2200, savings=100, savings_goal=1000, price=50)
#
# # Test 2: Price uses way more than half of disposable income -> RISKY
# run_test("Large purchase vs small disposable income", income=3000, fixed_expenses=2400, savings=500, savings_goal=2000, price=400)
#
# # Test 3: Affordable price, but savings goal progress is low -> THINK TWICE
# run_test("Low savings progress", income=3000, fixed_expenses=2000, savings=200, savings_goal=2000, price=150)
#
# # Test 4: Small price, strong savings progress -> GO FOR IT
# run_test("Comfortable purchase, strong savings", income=3000, fixed_expenses=2000, savings=1200, savings_goal=2000, price=150)


if __name__ == "__main__":
    main()

Welcome to the Smart Purchase Advisor!
Answer the following questions to see if your purchase is a smart move.

Enter your monthly income: $500
Enter your monthly fixed expenses: $100
Enter your current savings balance: $5000
Enter your savings goal amount: $10000
Enter the price of the item you want to buy: $1800

SMART PURCHASE ADVISOR - SUMMARY
Item Price:                   $   1800.00
Disposable Income:            $    400.00
Purchase % of Disposable Income:     450.0%
Savings Goal Progress:              50.0%
--------------------------------------------------
DECISION: NOT RECOMMENDED
--------------------------------------------------
This item costs $1800.00, which is more than your entire
  disposable income of $400.00.
  Advice: Hold off on this purchase entirely. Focus on increasing income
  or reducing fixed expenses before considering non-essential spending.



## Quick reflection

> Double-click to answer.

1. What did your first prompt miss that you had to add?
My earliest draft just said "build a purchase advisor that checks if someone can afford something." Running that mentally against the brief exposed three gaps:

No input validation rule, so a negative income or the word "lots" would have crashed the program or produced nonsense output.
No exact thresholds for the decision categories,  "afford" is not a number. I had to pin down the four bands (≤0/>disposable income, >50%, 25–50%, ≤25%+goal≥50%) and the order they're checked in, since an unordered set of conditions can overlap.
No instruction that each decision needed a genuinely different message. Without that, an AI will happily write one generic paragraph and just swap in the category label,  which fails the brief's "genuinely different decisions" requirement in spirit even if the classification logic is correct.
2. Which single detail made the biggest difference to the output?
Specifying the decision logic as a numbered, ordered checklist with exact percentage boundaries. Before that, the vaguer "warn if it's too expensive relative to income" version left the AI free to invent its own thresholds,  I tested a looser version and got a program that only produced two effective outcomes (basically "fine" and "not fine") because the boundaries it invented rarely triggered the middle cases.
3. Could a classmate run your final prompt cold and get a working tool? How do you know?
Yes, I verified it, not just assumed it: I ran the generated code through four test cases spanning all four decision categories and confirmed each produced the correct branch with a distinct, numbers-based message.A classmate pasting the same prompt should get functionally the same program.

## Before you submit

- [ ] Your final prompt is one complete block that needs no follow-up questions.
- [ ] The generated code runs top to bottom with no errors.
- [ ] The program makes different decisions for different inputs.
- [ ] You can explain what every part of your prompt is doing and why.
- [ ] You have downloaded the notebook and submitted it as your Week 3 lab ticket.

This is the first brick in a bigger wall. Over the semester your budget logic grows into a finance tracker, and the prompt-writing skill you practise here is one you will lean on the whole way.